# Handle Multiple Sequences

In this lab we will learn exactly how we can process and handle multiple sequences of different lengths and how we can increase how many sequences we can input to a model.

## Batch of inputs

Below we see that the code produces an error.

Why?

Well because transformers expect multiple sentences by default and we are giving it a **1D Tensor**

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = 'I have been waiting for this code my whole life!'

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)

model(input_ids)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (11) must match the size of tensor b (512) at non-singleton dimension 1

In [2]:
tokenized_inputs = tokenizer(sequence, return_tensors='pt')
print(tokenized_inputs)

{'input_ids': tensor([[ 101, 1045, 2031, 2042, 3403, 2005, 2023, 3642, 2026, 2878, 2166,  999,
          102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [3]:
input_ids = torch.tensor([ids])
print('Input Ids:', input_ids)

output = model(input_ids)
print('Logits:', output.logits)

Input Ids: tensor([[1045, 2031, 2042, 3403, 2005, 2023, 3642, 2026, 2878, 2166,  999]])
Logits: tensor([[-2.1184,  2.2083]], grad_fn=<AddmmBackward0>)


## Batching

What we did below is called batching where we can now feed it multiple sentences.

> "Using multiple sequences is just as simple as building a batch with a single sequence."

In [4]:
batch_ids = [ids, ids]

## Padding

Padding makes sure that all our sentences have the same length through out and is easier to process for out model

In [5]:
padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id]
]

In [6]:
sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200,200]]

batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
  ]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


## Attention Masks


The logits in the second row should be in the first layer tensor. This is because the attention layers in transformers are not ignoring the final padding tokens.

To mitigate that, we can tell the model what to "attend to" and what not to attend to.

In [7]:
attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
